# Parte 3: NDVI (Sentinel-2 / Landsat-8)
## Riesgo de Contaminación por Nitratos | La Rioja, 2015 - 2025

*Nota:* La Parte 3 es independiente. Solo enriquece el GeoDataFrame de puntos con NDVI anual.  

In [1]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
print('Librerías OK')

Librerías OK


In [2]:
BASE     = Path(r"C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja")
NDVI_DIR = BASE / '8_ndvi_sentinel2'
OUT_DIR  = BASE / '9_dataset_final'

# Input: GPKG con los 104 puntos RNIT de La Rioja + variables espaciales (salida P2)
RUTA_P2  = OUT_DIR / 'puntos_rnit_larioja_variables_espaciales_limpio.gpkg'

print('NDVI dir existe:', NDVI_DIR.exists())
print('Input P2 existe:', RUTA_P2.exists())
tifs = sorted(list(NDVI_DIR.glob('*.tif')) + list(NDVI_DIR.glob('*.tiff')))
print(f'GeoTIFFs NDVI encontrados: {len(tifs)}')
for t in tifs: print(' ', t.name)

NDVI dir existe: True
Input P2 existe: True
GeoTIFFs NDVI encontrados: 11
  ndvi_2015.tif
  ndvi_2016.tif
  ndvi_2017.tif
  ndvi_2018.tif
  ndvi_2019.tif
  ndvi_2020.tif
  ndvi_2021.tif
  ndvi_2022.tif
  ndvi_2023.tif
  ndvi_2024.tif
  ndvi_2025.tif


In [3]:
gdf = gpd.read_file(RUTA_P2)
print(f'Puntos cargados: {len(gdf)} | CRS: {gdf.crs}')
# Verificar que tiene columna ipa (clave de unión con P1)
assert 'ipa' in gdf.columns, 'Falta columna ipa'
print('Columnas:', gdf.columns.tolist())

Puntos cargados: 104 | CRS: EPSG:25830
Columnas: ['ipa', 'x_utm30', 'y_utm30', 'nombre_punto', 'municipio', 'provincia', 'masa_agua', 'red', 'lon_x', 'lat_x', 'lat_y', 'lon_y', 'soil_cec_0_5cm', 'soil_cec_15_30cm', 'soil_cec_5_15cm', 'soil_clay_0_5cm', 'soil_clay_15_30cm', 'soil_clay_5_15cm', 'soil_phh2o_0_5cm', 'soil_phh2o_15_30cm', 'soil_phh2o_5_15cm', 'soil_sand_0_5cm', 'soil_sand_15_30cm', 'soil_sand_5_15cm', 'soil_silt_0_5cm', 'soil_silt_15_30cm', 'soil_silt_5_15cm', 'soil_soc_0_5cm', 'soil_soc_15_30cm', 'soil_soc_5_15cm', 'soil_wv0033_0_5cm', 'soil_wv0033_15_30cm', 'soil_wv0033_5_15cm', 'soil_wv1500_0_5cm', 'soil_wv1500_15_30cm', 'soil_wv1500_5_15cm', 'clc2012_codigo', 'clc2012_nombre', 'clc2012_agricola', 'clc2018_codigo', 'clc2018_nombre', 'clc2018_agricola', 'cambio_clc_2012_2018', 'sigpac_uso', 'sigpac_uso_nombre', 'sigpac_superficie_m2', 'sigpac_pendiente_pct', 'sigpac_coef_regadio', 'sigpac_es_agricola', 'dist_sigpac_m', 'zona_vulnerable_nitratos', 'altitud_dem_m', 'geometr

### 1. Extracción de NDVI anual

Para cada uno de los 11 años disponibles, se extrae el valor de NDVI en la posición exacta de cada uno de los 104 puntos, descartando valores fuera del rango físico válido [-1, 1]. A partir de las 11 columnas anuales se calculan también cuatro variables resumen del período completo: media, mínimo, máximo y cambio entre el primer y el último año.

In [4]:
def extraer_ndvi(gdf_puntos: gpd.GeoDataFrame, dir_ndvi: Path) -> gpd.GeoDataFrame:
    """
    Extrae NDVI anual en los puntos RNIT.
    - Acepta archivos: ndvi_2018.tif, NDVI_S2_2018_LaRioja.tif, etc.
    - Filtra valores fuera de [-1, 1] → NaN
    - Calcula ndvi_media y ndvi_cambio sobre todos los años
    """
    tifs = sorted(list(dir_ndvi.glob('*.tif')) + list(dir_ndvi.glob('*.tiff')))
    if not tifs:
        print('Sin GeoTIFFs NDVI. Ver instrucciones GEE.')
        return gdf_puntos

    gdf = gdf_puntos.copy()
    print(f'{len(tifs)} archivos NDVI encontrados')

    for tif in tifs:
        match = re.search(r'(20\d{2})', tif.stem)
        if not match: continue
        anio = match.group(1)
        col  = f'ndvi_{anio}'

        try:
            with rasterio.open(tif) as src:
                gdf_t  = gdf.to_crs(src.crs)
                coords = [(g.x, g.y) for g in gdf_t.geometry]
                arr    = np.array([v[0] for v in src.sample(coords)], dtype=float)
                if src.nodata is not None:
                    arr[arr == src.nodata] = np.nan
                arr[(arr < -1) | (arr > 1)] = np.nan
            gdf[col] = arr
            print(f'  {col}: media={np.nanmean(arr):.3f} | nulos={np.isnan(arr).sum()}')
        except Exception as e:
            print(f'  Error {tif.name}: {e}')

    ndvi_cols = sorted([c for c in gdf.columns if re.fullmatch(r'ndvi_20\d{2}', c)])
    if len(ndvi_cols) >= 1:
        gdf['ndvi_media_periodo']  = gdf[ndvi_cols].mean(axis=1)
        gdf['ndvi_min_periodo']    = gdf[ndvi_cols].min(axis=1)
        gdf['ndvi_max_periodo']    = gdf[ndvi_cols].max(axis=1)
    if len(ndvi_cols) >= 2:
        gdf['ndvi_cambio_periodo'] = gdf[ndvi_cols[-1]] - gdf[ndvi_cols[0]]
        print('\nVariables derivadas calculadas: ndvi_media, ndvi_cambio')
    return gdf


gdf = extraer_ndvi(gdf, NDVI_DIR)
ndvi_cols = [c for c in gdf.columns if c.startswith('ndvi_')]
print(f'\nColumnas NDVI: {len(ndvi_cols)}')

11 archivos NDVI encontrados
  ndvi_2015: media=0.448 | nulos=0
  ndvi_2016: media=0.441 | nulos=0
  ndvi_2017: media=0.432 | nulos=0
  ndvi_2018: media=0.450 | nulos=0
  ndvi_2019: media=0.431 | nulos=0
  ndvi_2020: media=0.453 | nulos=0
  ndvi_2021: media=0.439 | nulos=0
  ndvi_2022: media=0.416 | nulos=0
  ndvi_2023: media=0.458 | nulos=0
  ndvi_2024: media=0.459 | nulos=0
  ndvi_2025: media=0.454 | nulos=0

Variables derivadas calculadas: ndvi_media, ndvi_cambio

Columnas NDVI: 15


### 2. Validación de los valores NDVI

Se revisan los estadísticos de cada año y se confirma que ningún valor quede fuera del rango [-1, 1], como control de calidad antes de guardar el resultado.

In [5]:
print('=== VALIDACIÓN NDVI ===')
ndvi_anuales = [c for c in gdf.columns if re.fullmatch(r'ndvi_20\d{2}', c)]
if ndvi_anuales:
    desc = gdf[ndvi_anuales].describe().round(3)
    display(desc)
    # Verificar rango
    for col in ndvi_anuales:
        mn, mx = gdf[col].min(), gdf[col].max()
        if (mn < -1) or (mx > 1):
            print(f'ERROR: {col} fuera de rango [{mn:.3f}, {mx:.3f}]')
    print('\nNDVI media La Rioja por año:')
    print(gdf[ndvi_anuales].mean().round(3))
print(f'\nShape final: {gdf.shape}')

=== VALIDACIÓN NDVI ===


,ndvi_2015,ndvi_2016,ndvi_2017,ndvi_2018,ndvi_2019,ndvi_2020,ndvi_2021,ndvi_2022,ndvi_2023,ndvi_2024,ndvi_2025
count,104.000,104.000,104.000,104.000,104.000,104.000,104.000,104.000,104.000,104.000,104.000
mean,0.448,0.441,0.432,0.450,0.431,0.453,0.439,0.416,0.458,0.459,0.454
std,0.189,0.204,0.232,0.225,0.230,0.238,0.233,0.245,0.244,0.242,0.246
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.325,0.280,0.256,0.291,0.279,0.276,0.271,0.217,0.269,0.281,0.255
50%,0.452,0.432,0.425,0.432,0.391,0.437,0.413,0.374,0.459,0.451,0.425
75%,0.598,0.587,0.623,0.637,0.612,0.647,0.613,0.608,0.665,0.666,0.678
max,0.871,0.876,0.883,0.852,0.881,0.889,0.916,0.896,0.894,0.925,0.918



NDVI media La Rioja por año:
ndvi_2015    0.448
ndvi_2016    0.441
ndvi_2017    0.432
ndvi_2018    0.450
ndvi_2019    0.431
ndvi_2020    0.453
ndvi_2021    0.439
ndvi_2022    0.416
ndvi_2023    0.458
ndvi_2024    0.459
ndvi_2025    0.454
dtype: float64

Shape final: (104, 68)


### 3. Guardado

In [6]:
ruta_gpkg = OUT_DIR / 'puntos_rnit_con_ndvi.gpkg'
ruta_csv  = OUT_DIR / 'puntos_rnit_con_ndvi.csv'

gdf.to_file(ruta_gpkg, driver='GPKG')
gdf.drop(columns='geometry', errors='ignore').to_csv(ruta_csv, index=False, encoding='utf-8-sig')

print('Guardado:', ruta_gpkg)
print(f'Puntos: {len(gdf)} | Columnas: {gdf.shape[1]}')
print(f'Columnas NDVI disponibles para Parte 4: {sorted([c for c in gdf.columns if "ndvi" in c])}')
print('\nPróximo paso: ejecutar Parte 4 (integración final)')

Guardado: C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja\9_dataset_final\puntos_rnit_con_ndvi.gpkg
Puntos: 104 | Columnas: 68
Columnas NDVI disponibles para Parte 4: ['ndvi_2015', 'ndvi_2016', 'ndvi_2017', 'ndvi_2018', 'ndvi_2019', 'ndvi_2020', 'ndvi_2021', 'ndvi_2022', 'ndvi_2023', 'ndvi_2024', 'ndvi_2025', 'ndvi_cambio_periodo', 'ndvi_max_periodo', 'ndvi_media_periodo', 'ndvi_min_periodo']

Próximo paso: ejecutar Parte 4 (integración final)


**Observación:**

El NDVI se trata como un atributo del punto, igual que el suelo, pero con una dimensión temporal: cada punto tiene 11 valores, uno por año, y es Parte 4 la que decide cuál de esos 11 le corresponde a cada observación según su fecha.